## 1. Imports & Core Dependencies
Loads all required libraries for data loading (parquet), feature handling, cross-validation, modelling (XGBoost), and hyperparameter tuning (Optuna).

In [2]:
#Imports 
import os, json, datetime
import numpy as np
import pandas as pd
import fastf1 as ff1

from pathlib import Path
import datetime


## 2. Paths and Environment Sanity Checks
Defines dataset paths and verifies the notebook can read inputs and write outputs reliably (prevents silent path bugs).

In [3]:
#Path Checker Cell

OUT_DIR   = "../../data_processed"

OUT_DIR = Path(OUT_DIR)   # <-- this is the missing line
testp_file = OUT_DIR / "_parquets_path_test.txt"
with open(testp_file, "w") as f:
    f.write(f"Parquets test written at {datetime.datetime.now().isoformat()}\n")
    f.write(f"Works from the ml notebook fool\n")


print("Test file written to:", testp_file)


Test file written to: ..\..\data_processed\_parquets_path_test.txt


## 3. Load Pre-Qualifying Dataset (2018–2025) and Inspect Coverage
Loads the consolidated dataset and checks:
- shape (rows/columns)
- seasons present (2018–2025)
- sample counts per year
This confirms the parquet contains the expected time span before splitting.

In [9]:
#Checks DF shape
from pathlib import Path
import pandas as pd


DATASET_PATH = OUT_DIR / "preq_dataset.parquet"

df = pd.read_parquet(DATASET_PATH)
print(df.shape)

(3456, 16)


## 4. Data Integrity and Leakage Checks (Pre-Split)
Validates key assumptions before modelling:
- required columns exist (year, round, target, etc.)
- no null explosions in critical fields
- no “quali-derived” columns are mistakenly included as features
This prevents leakage and downstream training crashes.

In [ ]:
#remove invalid data,
df = df[df["quali_position"].notna()].reset_index(drop=True)

In [8]:
df

,year,round,event_name,country,location,driver,n_fp_laps,n_fp_sessions_participated,best_fp_lap_time_overall,avg_fp_lap_time,median_fp_lap_time,best_last_fp_lap_time,best_last_fp_s1,best_last_fp_s2,best_last_fp_s3,quali_position
0,2018,1,Australian Grand Prix,Australia,Melbourne,ALO,43,3,85.200,98.581512,96.2460,94.298,31.144,25.268,37.886,11.0
1,2018,1,Australian Grand Prix,Australia,Melbourne,BOT,63,3,84.159,98.692540,94.1740,94.174,31.326,25.150,37.698,10.0
2,2018,1,Australian Grand Prix,Australia,Melbourne,ERI,61,3,86.814,100.017279,90.7760,88.890,29.842,23.687,35.361,17.0
3,2018,1,Australian Grand Prix,Australia,Melbourne,GAS,69,3,85.945,99.882739,91.8910,94.990,31.187,25.521,38.282,20.0
4,2018,1,Australian Grand Prix,Australia,Melbourne,GRO,54,3,84.648,96.055611,90.5690,96.171,31.767,25.553,38.851,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3451,2025,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,RUS,55,3,83.334,104.099691,91.5630,83.334,17.055,36.082,30.197,4.0
3452,2025,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,SAI,61,3,83.811,104.445508,90.2330,83.811,17.136,36.099,30.576,12.0
3453,2025,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,STR,34,2,83.832,107.180706,90.8205,83.895,17.237,36.237,30.421,15.0
3454,2025,24,Abu Dhabi Grand Prix,United Arab Emirates,Yas Island,TSU,25,2,84.303,99.908240,90.4320,84.693,17.328,36.361,30.970,10.0


## Cell 6 — Define Target Variable and Non-Feature Columns
Defines the prediction target and the list of columns to exclude from
the feature matrix (identifiers, metadata, and leakage-prone fields).

In [12]:
TARGET = "quali_position"
DROP_COLS = ["year","round","event_name","country","location","driver"]

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

## Cell 8 — Construct Feature Matrices (X) and Targets (y)
Builds X and y for each split by:
- removing non-feature columns
- separating inputs from the prediction target

Ensures that X contains only pre-qualifying information.

In [18]:
import numpy as np
import pandas as pd

TARGET = "quali_position"

# force numeric
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

# drop bad labels (NaN/inf)
mask = df[TARGET].notna() & np.isfinite(df[TARGET].to_numpy())
bad = (~mask).sum()

print("Dropping bad labels:", bad)
if bad:
    print(df.loc[~mask, ["year","round","driver", TARGET]].head(20))

df = df.loc[mask].reset_index(drop=True)

# rebuild X/y consistently AFTER filtering
DROP_COLS = ["year","round","event_name","country","location","driver"]
X = df.drop(columns=DROP_COLS + [TARGET], errors="ignore")
y = df[TARGET].astype(float)

print("Final shapes:", X.shape, y.shape)
print("Any NaN in y?", y.isna().any())
print("Any inf in y?", np.isinf(y.to_numpy()).any())

Dropping bad labels: 4
      year  round driver  quali_position
460   2019      3    ALB             NaN
463   2019      3    GIO             NaN
1267  2021      5    MSC             NaN
3380  2025     21    BOR             NaN
Final shapes: (3452, 9) (3452,)
Any NaN in y? False
Any inf in y? False


## Cell 10 — Baseline Leave-One-Year-Out (LOYO) Cross-Validation
Evaluates a baseline XGBoost model using LOYO cross-validation
on the training seasons to establish a performance reference.

In [19]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

groups = df["year"]   # LOYO by year

gkf = GroupKFold(n_splits=len(groups.unique()))

maes = []

for train_idx, test_idx in gkf.split(X, y, groups):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    )

    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    maes.append(mean_absolute_error(yte, preds))

print("Baseline LOYO MAE:", sum(maes)/len(maes))

Baseline LOYO MAE: 4.848688658323019


We now do hyper param tuning with optuna

In [20]:
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective": "reg:squarederror",
        "random_state": 42,
    }

    maes = []

    for tr, te in gkf.split(X, y, groups):
        model = XGBRegressor(**params)
        model.fit(X.iloc[tr], y.iloc[tr])
        preds = model.predict(X.iloc[te])
        maes.append(mean_absolute_error(y.iloc[te], preds))

    return sum(maes) / len(maes)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)

c:\Users\aadha\anaconda3\envs\f1ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-01 18:33:29,839] A new study created in memory with name: no-name-54bf38fb-3f6f-46c2-ba74-860387025f14
[I 2026-03-01 18:33:31,008] Trial 0 finished with value: 4.8103929684869815 and parameters: {'n_estimators': 290, 'max_depth': 4, 'learning_rate': 0.029606217588518384, 'subsample': 0.7695182911833081, 'colsample_bytree': 0.6460081775203739}. Best is trial 0 with value: 4.8103929684869815.
[I 2026-03-01 18:33:33,798] Trial 1 finished with value: 4.794969207953594 and parameters: {'n_estimators': 422, 'max_depth': 6, 'learning_rate': 0.022748694838078065, 'subsample': 0.7950773704785393, 'colsample_bytree': 0.7913154958747578}. Best is trial 1 with value: 4.794969207953594.
[I 2026-03-01 18:33:34,992] Trial 2 finished wit

Best MAE: 4.765465683321537
Best params: {'n_estimators': 526, 'max_depth': 6, 'learning_rate': 0.011940483407008439, 'subsample': 0.6267753620848954, 'colsample_bytree': 0.8100112174446987}


In [21]:
TARGET = "quali_position"
KEY_COLS = ["year","round","event_name","country","location","driver"]

X = df.drop(columns=KEY_COLS + [TARGET])
y = df[TARGET].astype(float)

In [22]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

def eval_loyo_mae(X, y, groups, params):
    gkf = GroupKFold(n_splits=len(np.unique(groups)))
    fold_mae = {}

    for tr, te in gkf.split(X, y, groups):
        yr = int(pd.Series(groups).iloc[te].unique()[0])

        model = XGBRegressor(**params)
        model.fit(X.iloc[tr], y.iloc[tr])
        pred = model.predict(X.iloc[te])

        mae = mean_absolute_error(y.iloc[te], pred)
        fold_mae[yr] = mae

    return fold_mae

In [23]:
best_params = study.best_params

groups = df["year"].values

fold_mae = eval_loyo_mae(X, y, groups, best_params)
print("LOYO MAE by year:")
for yr in sorted(fold_mae):
    print(yr, "->", fold_mae[yr])

print("Mean LOYO MAE:", sum(fold_mae.values()) / len(fold_mae))

LOYO MAE by year:
2018 -> 4.284294551894778
2019 -> 4.125139845044989
2020 -> 4.912986889701326
2021 -> 4.744562669332587
2022 -> 5.250198843262412
2023 -> 5.020119772654146
2024 -> 4.805215508056832
2025 -> 5.037222078797215
Mean LOYO MAE: 4.772467519843035


In [24]:
final_model = XGBRegressor(**best_params)
final_model.fit(X, y)

# feature importance (gain-based)
importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(15))

best_last_fp_lap_time         0.137628
best_last_fp_s1               0.131432
best_last_fp_s2               0.128596
best_last_fp_s3               0.125885
best_fp_lap_time_overall      0.117317
median_fp_lap_time            0.104116
n_fp_sessions_participated    0.096584
avg_fp_lap_time               0.091647
n_fp_laps                     0.066794
dtype: float32
